# 04 — Felt connectedness as a facet of the collective climate

Between/within decomposition of connectedness on belief drift; correlation of tone and connectedness; joint (group-level) model.

**Expected:** between-group β=+15.96, 95%CI[4.10,27.82], p=.008; within-person β=−2.33, p=.38; ICC=0.25. Tone–connectedness r=.57, 95%CI[.10,.83], p=.022 (16 groups). Joint (n=16): tone β=+4.96, p=.12; connectedness β=+3.95, p=.20. Baseline-dispersion control: connectedness~dispersion r=−.38 p=.12 (18 groups); dispersion→shift r=−.22 p=.37; between-group connectedness b=+15.96 p=.008 → +15.75 p=.020 controlled.

In [1]:
import pandas as pd, numpy as np, warnings
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
DATA = '../data'
def load_belief(): return pd.read_excel(f'{DATA}/belief_items.xlsx')
def load_subject(): return pd.read_excel(f'{DATA}/subject_measures.xlsx')
def load_group(): return pd.read_csv(f'{DATA}/group_conversation.csv')
def rep(m,t):
    ci=m.conf_int().loc[t]
    return f'b={m.params[t]:+.3f} 95%CI[{ci[0]:+.3f}, {ci[1]:+.3f}] p={m.pvalues[t]:.4f}'
def corr_ci(x,y):
    r,p=stats.pearsonr(x,y); n=len(x); z=np.arctanh(r); se=1/np.sqrt(n-3)
    return r,np.tanh(z-1.96*se),np.tanh(z+1.96*se),p,n


In [2]:
s=load_subject()
c=s[s.condition=='collaborative'][['group','conn_mean','bel_related_change']].dropna()
print('connectedness subjects:',len(c),'groups:',c.group.nunique())
c['gm']=c.groupby('group').conn_mean.transform('mean'); c['wi']=c.conn_mean-c.gm
m=smf.mixedlm('bel_related_change ~ gm + wi', c, groups=c.group).fit()
print('between-group:', rep(m,'gm'))
print('within-person:', rep(m,'wi'))
vc=m.cov_re.iloc[0,0]; print(f'ICC = {vc/(vc+m.scale):.2f}')


connectedness subjects: 72 groups: 18
between-group: b=+15.961 95%CI[+4.100, +27.822] p=0.0084
within-person: b=-2.328 95%CI[-7.571, +2.915] p=0.3842
ICC = 0.25


In [3]:
# tone-connectedness correlation and joint model (group level)
g=load_group(); gc=s[s.condition=='collaborative'].groupby('group').conn_mean.mean()
b=load_belief().dropna(subset=['pre','post']); e=b[(b.condition=='collaborative')&(b.topic=='related')].copy(); e['ch']=e.post-e.pre
gd=e.groupby('group')['ch'].mean()
G=g.set_index('group')[['tone']].join(gc.rename('conn')).join(gd.rename('drift')).dropna()
r,lo,hi,p,n=corr_ci(G.tone,G.conn); print(f'tone-connectedness r={r:+.2f} 95%CI[{lo:+.2f},{hi:+.2f}] p={p:.3f} n={n}')
for c2 in ['tone','conn']: G[c2+'_z']=(G[c2]-G[c2].mean())/G[c2].std()
mj=smf.ols('drift ~ tone_z + conn_z', G).fit()
print('joint tone:', rep(mj,'tone_z')); print('joint connectedness:', rep(mj,'conn_z'))


tone-connectedness r=+0.57 95%CI[+0.10,+0.83] p=0.021 n=16
joint tone: b=+4.946 95%CI[-1.428, +11.319] p=0.1175
joint connectedness: b=+3.944 95%CI[-2.430, +10.317] p=0.2043


In [4]:
# baseline-dispersion control for connectedness (Results, second climate feature)
s=load_subject(); b=load_belief().dropna(subset=['pre','post'])
e=b[(b.condition=='collaborative')&(b.topic=='related')]; piv=e.pivot_table('pre','Item','participant')
disp={}
for grp,mem in e.groupby('group')['participant'].unique().items():
    m=[x for x in mem if x in piv.columns]
    ds=[np.nanmean(np.abs(piv[m[i]]-piv[m[j]])) for i in range(len(m)) for j in range(i+1,len(m))]
    disp[grp]=np.mean(ds) if ds else np.nan
disp=pd.Series(disp, name='disp_pre')
c=s[s.condition=='collaborative'].dropna(subset=['conn_mean','bel_related_change']).join(disp,on='group').dropna(subset=['disp_pre'])
G=c.groupby('group').agg(conn=('conn_mean','mean'), rel=('bel_related_change','mean'), disp=('disp_pre','first'))
print(f'groups: {len(G)} | participants: {len(c)}')
r,lo,hi,p,n=corr_ci(G.conn,G.disp); print(f'  connectedness ~ baseline dispersion  r={r:+.3f} 95%CI[{lo:+.2f}, {hi:+.2f}] p={p:.3f} n={n}')
r,lo,hi,p,n=corr_ci(G.disp,G.rel);  print(f'  dispersion ~ related shift           r={r:+.3f} 95%CI[{lo:+.2f}, {hi:+.2f}] p={p:.3f}')
c['conn_grp']=c.groupby('group').conn_mean.transform('mean'); c['conn_wi']=c.conn_mean-c.conn_grp
base=smf.mixedlm('bel_related_change ~ conn_grp + conn_wi', c, groups=c.group).fit()
ctrl=smf.mixedlm('bel_related_change ~ conn_grp + conn_wi + disp_pre', c, groups=c.group).fit()
print('  between-group connectedness, no control  :', rep(base,'conn_grp'))
print('  between-group connectedness, | dispersion:', rep(ctrl,'conn_grp'))
print('  baseline dispersion                      :', rep(ctrl,'disp_pre'))

groups: 18 | participants: 72
  connectedness ~ baseline dispersion  r=-0.377 95%CI[-0.72, +0.11] p=0.123 n=18
  dispersion ~ related shift           r=-0.224 95%CI[-0.63, +0.27] p=0.371
  between-group connectedness, no control  : b=+15.961 95%CI[+4.100, +27.822] p=0.0084
  between-group connectedness, | dispersion: b=+15.752 95%CI[+2.527, +28.976] p=0.0196
  baseline dispersion                      : b=-0.026 95%CI[-0.643, +0.592] p=0.9345
